# 050 — Set up MSA runs for the 3s & 5s mdof case-study structures

Wires the Multiple-Stripe-Analysis (MSA) run files into the **existing** per-site
`mdof/` analysis folders built by notebook 011, for **both** the 3-storey and
5-storey structures, at `DEST_ROOT/site_{ii}/{n}s/mdof/`
(`DEST_ROOT = wp1_casestudy_sites`).

Each structure gets only the stripe selection pickles for **its own run list**,
built from `data_processed/05_gcim_distributions/imls_for_selection_AvgSA_03.json`:
its four selected `stripes`, plus every `additional` IML notebook 060 added to close
a fragility tail, plus only those `reserves` 060 flagged as worth running in
`reserve_stripes_to_run_AvgSA_03.json`. The remaining reserves are spare capacity
that was never needed and are deliberately left out.

This notebook does **not** design structures or run modal analyses. It only adds
the MSA files (the coordinator + worker, injection/recorder helpers, and one
`config_msa_{IM}.py` + `{IM}/` stripe-pickle folder), then writes **per-storey
multi-building launchers** (one for 3s, one for 5s).

Pipeline (mirrors IDA): `run_batch_msa_buildings.py` (one coordinator per building,
shared semaphore) → `run_batch_msa_per_record.py` (dispatches every `(stripe, record)`
pair at once) → `run_msa_per_record.py` worker. Runs are **resumable** — re-running a
launcher skips `(stripe, record)` pairs already analysed, and stripe result folders are
intensity-keyed (`stripe_{iml}`), so an extra stripe can be added and re-run later.

## Dependencies

Run notebook 011 first: it builds the `mdof/` folders and their `structural_model.py`.
Any (site, storey) whose `structural_model.py` is missing is **warned and skipped**
(and excluded from the batch files).

## Filtering

Set `FILTER_TO_COMPLETE_SITES` at the top: when `True` the batch launchers list only
the sites flagged complete by the record-availability audit
(`data_processed/06_gm_selection/complete_site_im_sets.csv`). The MSA files are
copied into every existing folder regardless — the filter only affects the batch
listings.

In [7]:
# %load_ext autoreload
# %autoreload 2

## 0. Setup & parameters

In [8]:
import json
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_file,
    copy_analysis_config,
    copy_batch_ida_buildings,
)
from phd_project.scripts.WP1_ground_motion_set.gm_selection import stripe_pickle_path

cfg = config.load_config()

# --- parameters -----------------------------------------------------------
DEST_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]   # site_{ii}/{n}s/mdof/
GM_SET = "AvgSA_03"
STOREYS = [3, 5]                     # build MSAs for both the 3s and 5s structures
STRIPE_ORDER_ASCENDING = True
MAX_N_RECORDS = None                 # cap records per stripe (None = all)

# When True the per-storey batch launchers list only the completed sites (from
# complete_site_im_sets.csv). MSA files are copied into every existing folder
# regardless; this only affects the batch listings.
FILTER_TO_COMPLETE_SITES = True

# mdof EDP recorder template (roof drift)
RECORDER_KEY = "ida_process_recorder_roof&maxstorey_drift"

RECORD_FOLDER = Path(cfg["results"][f"{GM_SET}_record_selection"])
IMLS_JSON = Path(cfg["proc_data"][f"{GM_SET}_imls_for_selection"])
# Reserve stripes notebook 060 found worth running. Absent until 060 has been run, in
# which case no reserves are set up and only the originally selected stripes go in.
RESERVE_STRIPES_JSON = Path(cfg["proc_data"][f"{GM_SET}_reserve_stripes"])
# Batch launchers land under BATCH_BASE/{n}s/mdof/ (matches nb 011); the folder
# path encodes the storeys + system, so the filename is just msa_{IM}.py.
BATCH_BASE = Path(cfg["scripts"]["batch_run_analyses"]) / "casestudy_sites"
print(f"DEST_ROOT = {DEST_ROOT}")

DEST_ROOT = D:\07_wp1_casestudy_sites


## 1. Read the completed-site list

`ready_sites` — the site indices flagged complete for `GM_SET` by the
record-availability audit. Only used to filter the batch launchers when
`FILTER_TO_COMPLETE_SITES` is `True`.

In [9]:
ready_csv = Path(cfg["proc_data"]["gm_selection"]) / "complete_site_im_sets.csv"
ready_df = pd.read_csv(ready_csv)

ready_sites: set[int] = {
    int(row["site"]) for _, row in ready_df.iterrows() if str(row["im"]) == GM_SET
}

print(f"{len(ready_sites)} completed sites for {GM_SET}  (from {ready_csv.name})")

57 completed sites for AvgSA_03  (from complete_site_im_sets.csv)


## 2. Per-structure stripes to run

The per-structure entry comes from `imls_for_selection_{GM_SET}.json`, keyed by site
index then structure tag `"{n}s_cbf_dc2_site{ii}"`, with three fields written by
notebook 017: `stripes` (the four originally selected IMLs), `reserves`
(`[low, high]`, either sometimes `null`) and `additional` (remediation IMLs from
notebook 060).

`stripes_to_run` combines them: every stripe, every additional IML, and only those
reserves listed in `reserve_stripes_to_run_{GM_SET}.json`. The sites to set up are
the JSON's site keys.

In [10]:
with open(IMLS_JSON) as f:
    imls_data = json.load(f)

# sites to set up = the site keys in the IML json
all_sites = sorted(int(k) for k in imls_data)


def structure_tag(site_idx: int, n: int) -> str:
    return f"{n}s_cbf_dc2_site{site_idx}"


def load_reserve_stripes() -> dict[tuple[int, int], set[float]]:
    """Reserves notebook 060 flagged as worth running, as {(site, storeys): {imls}}.

    Stored as {"3": [[site, iml], ...], "5": [...]}. Absent until 060 has run, which is
    not an error - it just means no reserve stripes are set up.
    """
    if not RESERVE_STRIPES_JSON.is_file():
        print(f"{RESERVE_STRIPES_JSON.name} not found - no reserve stripes will be set up")
        return {}
    with open(RESERVE_STRIPES_JSON) as f:
        raw = json.load(f)
    wanted: dict[tuple[int, int], set[float]] = {}
    for n, pairs in raw.items():
        for site, iml in pairs:
            wanted.setdefault((int(site), int(n)), set()).add(round(float(iml), 4))
    return wanted


reserve_stripes = load_reserve_stripes()


def stripes_to_run(site_idx: int, n: int) -> list[float]:
    """Every IML this structure's MSA should cover, ascending.

    The four originally selected stripes, plus every additional IML notebook 060 added to
    close a fragility tail, plus only those reserves 060 flagged as worth running - the
    rest of the reserves are spare capacity that was never needed, and setting them up
    would mean 30 wasted analyses each.
    """
    entry = imls_data[str(site_idx)][structure_tag(site_idx, n)]
    wanted = set(entry["stripes"]) | set(entry["additional"])
    wanted |= ({x for x in entry["reserves"] if x is not None}
               & reserve_stripes.get((site_idx, n), set()))
    return sorted(wanted)


print(f"{len(all_sites)} sites in {IMLS_JSON.name}, "
      f"{sum(len(v) for v in reserve_stripes.values())} reserve stripes requested")
for n in STOREYS:
    example = stripes_to_run(all_sites[0], n)
    print(f"  {n}s example (site {all_sites[0]}): {example}")

60 sites in imls_for_selection_AvgSA_03.json
  3s example (site 0): [0.31, 0.36, 0.4, 0.5]
  5s example (site 0): [0.285, 0.33, 0.36, 0.4]


## 3. `add_msa_files` helper

Copies the MSA coordinator (`run_batch_msa_per_record.py`) and its worker
(`run_msa_per_record.py`) into a `mdof/` folder, along with the injection/recorder
helpers, and creates one `config_msa_{IM}.py` + `{IM}/` stripe-pickle subfolder holding
only the structure's **middle 4** stripe pickles (resolved via `stripe_pickle_path`).
Returns any expected-but-missing stripe pickles so they can be reported.

In [11]:
def add_msa_files(folder: Path, site_idx: int, n: int) -> list[Path]:
    """Add the MSA coordinator/worker/injection/recorder files + one config and a
    stripe-pickle subfolder holding this structure's full run list. Returns the
    list of expected stripe pickles that were missing at source."""
    copy_file(cfg["templates"]["run_batch_msa_per_record"],
              folder / "run_batch_msa_per_record.py")
    copy_file(cfg["templates"]["run_msa_per_record"],
              folder / "run_msa_per_record.py")
    copy_file(cfg["templates"]["nltha_injection_update_damping"],
              folder / "injection_functions.py")
    copy_file(cfg["templates"][RECORDER_KEY],
              folder / "msa_process_recorders.py")

    record_src = Path(cfg["proc_data"]["gm_records"]).as_posix()

    gm_dir = folder / f"{GM_SET}_stripes"
    gm_dir.mkdir(parents=True, exist_ok=True)
    # clear any previously-copied stripe pickles first so stale / old-scheme names
    # don't linger alongside the current run list (find_stripe_pickles globs
    # all *stripe*gm_selection* in the folder). The folder is rebuilt complete, with
    # the already-analysed stripes as well as the new ones, so it always describes the
    # whole MSA rather than just the outstanding work.
    for old in gm_dir.glob("*__stripe_*__gm_selection.pickle"):
        old.unlink()

    missing: list[Path] = []
    for iml in stripes_to_run(site_idx, n):
        src = stripe_pickle_path(RECORD_FOLDER, site_idx, iml)
        if src.exists():
            copy_file(src, gm_dir / src.name)
        else:
            missing.append(src)

    copy_analysis_config(
        cfg["templates"]["config_msa"],
        folder / f"config_msa_{GM_SET}.py",
        results_folder_name=f"msa_{GM_SET}",
        model_file_name = "initialise_model_idamsa.py",
        gm_selection_src_str=gm_dir.as_posix(),
        record_src_str=record_src,
        stripe_order_ascending=STRIPE_ORDER_ASCENDING,
        max_n_records=MAX_N_RECORDS,
    )
    return missing

## 4. Add MSA files to the 3s & 5s mdof folders

For every site and each storey, add the MSA files to the `mdof/` folder built by
notebook 011. Missing folders / structural models are warned and skipped (and are
excluded from the batch files). Files are added for every existing folder,
regardless of the completed-site filter.

In [12]:
# storey -> {site_idx: folder}, only (site, storey) successfully set up (model present)
built: dict[int, dict[int, Path]] = {n: {} for n in STOREYS}
skipped: list[tuple[int, int]] = []
missing_stripes: list[Path] = []

for site_idx in all_sites:
    for n in STOREYS:
        folder = DEST_ROOT / f"site_{site_idx}" / f"{n}s" / "mdof"
        if not (folder / "config_structural_model.py").exists():
            print(f"WARNING: skipping site {site_idx} [{n}s]: "
                  f"{folder / 'config_structural_model.py'} not found (run nb 011 first)")
            skipped.append((site_idx, n))
            continue
        missing_stripes += add_msa_files(folder, site_idx, n)
        built[n][site_idx] = folder

for n in STOREYS:
    print(f"{n}s: MSA files added for {len(built[n])} sites")
if skipped:
    print(f"skipped {len(skipped)} (site, storey) pair(s): {skipped}")
if missing_stripes:
    print(f"WARNING: {len(missing_stripes)} expected stripe pickle(s) missing at source, "
          f"e.g. {missing_stripes[0].name}")

3s: MSA files added for 60 sites
5s: MSA files added for 60 sites


## 5. Write the per-storey multi-building launchers

One launcher per storey at `BATCH_BASE/{n}s/mdof/msa_{IM}.py` (e.g.
`.../3s/mdof/msa_AvgSA03.py`) — the folder path encodes the storeys + system, so the
filename is just the analysis type. Each is a `run_batch_msa_buildings` launcher whose
`buildings` list names a built site's `mdof/` folder + its `config_msa_{IM}.py`; the
coordinator (`run_batch_msa_per_record.py`) already lives in each folder. When
`FILTER_TO_COMPLETE_SITES` is `True`, only sites in `ready_sites` are listed.
`copy_batch_ida_buildings` serialises the buildings list into the runnable launcher (one
coordinator window per building, workers drawn from the shared semaphore).

In [13]:
batch_name = f"msa_{GM_SET.replace('_', '')}.py"   # -> msa_AvgSA03.py

for n in STOREYS:
    msa_buildings = [
        {"folder": folder, "config": folder / f"config_msa_{GM_SET}.py"}
        for site_idx, folder in sorted(built[n].items())
        if not FILTER_TO_COMPLETE_SITES or site_idx in ready_sites
    ]
    batch_dir = BATCH_BASE / f"{n}s" / "mdof"
    batch_dir.mkdir(parents=True, exist_ok=True)
    batch_dst = batch_dir / batch_name
    copy_batch_ida_buildings(
        cfg["templates"]["run_batch_msa_buildings"], batch_dst, msa_buildings)
    filt = "completed sites only" if FILTER_TO_COMPLETE_SITES else "all built sites"
    print(f"wrote {batch_dst.relative_to(BATCH_BASE)}: "
          f"{len(msa_buildings)} buildings ({filt})")

wrote 3s\mdof\msa_AvgSA03.py: 57 buildings (completed sites only)
wrote 5s\mdof\msa_AvgSA03.py: 57 buildings (completed sites only)
